# 01 — Explore sources

Read-only scan of the four `IngestSource`s from Phase A. Use this to:

- Sanity-check that source DBs are reachable and contain the expected shapes
- Get item counts per source (informs `--limit` choices for eval runs)
- Eyeball content lengths to inform chunker / chunk_size decisions
- Sample a handful of items per source for hand-curation in notebook 02

In [ ]:
import os
from pathlib import Path

import pandas as pd

from domains.notes.sources import LocalFileSource
from domains.raw_store.sources import RawStoreSource
from domains.research.sources import ResearchSource
from domains.sessions.sources import SessionsSource

BACKUP = Path(os.environ.get("BACKUP_SOURCE_DIR", "~")).expanduser()
BACKUP

In [2]:
# Build sources. Skip any whose path is missing.
sources = {}
if (p := BACKUP / "raw_store.db").exists():
    sources["raw_store"] = RawStoreSource(p)
if (p := BACKUP / "sessions.db").exists():
    sources["sessions"] = SessionsSource(p)
if (p := BACKUP / "research.db").exists():
    sources["research"] = ResearchSource(p)
if (p := BACKUP / "notes").is_dir():
    sources["notes"] = LocalFileSource(p)
list(sources)

['raw_store', 'sessions', 'research', 'notes']

In [6]:
# Item counts per source.
counts = {name: len(s.get_item_ids()) for name, s in sources.items()}
pd.Series(counts, name="n_items")

raw_store    949
sessions     184
research       7
notes         49
Name: n_items, dtype: int64

In [7]:
# Content-length distribution per source — chars per item. Informs chunk_size choice.
frames = []
for name, s in sources.items():
    items = s.get_items()
    frames.append(
        pd.DataFrame(
            {"source": name, "chars": [len(item.text) for item in items]}
        )
    )
lengths = pd.concat(frames, ignore_index=True)
lengths.groupby("source")["chars"].describe()

,count,mean,std,min,25%,50%,75%,max
source,,,,,,,,
notes,49.0,7745.346939,7685.553294,177.0,1994.0,5086.0,11747.00,31531.0
raw_store,949.0,9726.394099,20824.941402,0.0,0.0,0.0,11868.00,233492.0
research,7.0,8702.571429,6073.670797,3610.0,4458.0,5584.0,11396.50,20015.0
sessions,184.0,8638.326087,8910.934143,0.0,2940.0,6357.0,11354.25,57263.0


In [8]:
# Sample 3 items per source for visual inspection.
for name, s in sources.items():
    print(f"\n=== {name} ===")
    for item in s.get_items()[:3]:
        print(f"  {item.item_id}: {item.title!r} ({len(item.text)} chars)")


=== raw_store ===
  medium::https://medium.com/@alirezarezvani/i-took-boris-cherny-the-creator-of-claude-code-at-his-word-here-is-what-4-days-of-following-his-1b660da12400: 'I Took Boris Cherny (the Creator of Claude Code) at His Word — Here Is What 4…' (0 chars)
  medium::https://medium.com/@intellizab/the-ai-fomo-trap-why-95-of-workers-are-faking-it-f9d70df8b007: 'The AI FOMO Trap: Why 95% of Workers Are Faking It' (0 chars)
  medium::https://medium.com/@shethpriyanka001/i-replaced-my-entire-etl-pipeline-with-microsoft-fabric-heres-what-happened-e4d78a03a695: 'I Replaced My Entire ETL Pipeline with Microsoft Fabric — Here’s What Happened' (0 chars)

=== sessions ===
  d803cab8-088f-4536-80e8-4b2ad22699f6: 'The user requested to review the latest Medium Daily Digest newsletter, particularly focusing on an article about Anthro' (1373 chars)
  27859f9b-a0ee-46dd-aa41-48a81b94d042: 'During the session, the user and AI Newsletter Assistant discussed the latest Medium Daily Digest newslet